# ComfyUI-PlatoKeys — Qwen3.5-2B → Qwen3.5-9B Calibration (MOSS-Image)

Closed-form cross-encoder conditioning alignment inspired by Stanford Hazy Research's [**Plato's Key**](https://hazyresearch.stanford.edu/blog/2026-11-25-platos-key).

This notebook streams ~3,000 diverse image prompts (plus MOSS-Image typography, tag, bilingual, layout JSON, and negative prompts) through **Qwen3.5-2B** and **Qwen3.5-9B** sequentially, accumulates exact `float64` sufficient statistics, and fits & compares 4 closed-form maps in seconds:

1. **`procrustes`** — exact Plato's Key (semi-orthogonal rotation $R = UV^\top$ + least-squares global scale $s$)
2. **`procrustes_norm`** — semi-orthogonal rotation $R$ + target-variance-matching scale
3. **`ridge`** — closed-form Ridge regression (auto-tuned $\alpha$ on held-out validation prompts)
4. **`ridge_norm`** — closed-form Ridge + target-variance-matching scale

**Recommended Colab GPU:** **L4 (24 GB)** (~3–5 min) or **A100 (40/80 GB)** (~1.5–2 min). Free **T4 (16 GB)** works automatically via 8-bit target loading (`--target-8bit`).

In [ ]:
#@title 1. Check GPU & install dependencies
import torch

assert torch.cuda.is_available(), "Switch runtime to GPU (Runtime -> Change runtime type -> L4 / A100 / T4)."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
bf16_ok = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM, bf16={bf16_ok})")

NEED_8BIT = vram_gb < 20.0
if NEED_8BIT:
    print("VRAM < 20 GB detected -> will install bitsandbytes and load Qwen3.5-9B in 8-bit.")

!pip -q install -U "transformers>=4.51.0" safetensors datasets accelerate
if NEED_8BIT:
    !pip -q install -U bitsandbytes

In [ ]:
#@title 2. Clone or update ComfyUI-PlatoKeys
import os

REPO_URL = "https://github.com/Nynxz/ComfyUI-PlatoKeys.git"  #@param {type:"string"}
REPO_DIR = "/content/ComfyUI-PlatoKeys"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd /content/ComfyUI-PlatoKeys

In [ ]:
#@title 3. Configure calibration & run
import os
import subprocess

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SOURCE_MODEL = "Qwen/Qwen3.5-2B"  #@param {type:"string"}
TARGET_MODEL = "Qwen/Qwen3.5-9B"  #@param {type:"string"}
PRESET = "moss_image"  #@param ["moss_image", "qwen35_default", "raw_last"]
NUM_PROMPTS = 2000  #@param {type:"integer"}
VAL_PROMPTS = 150  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
MAX_LENGTH = 256  #@param {type:"integer"}
ALSO_REVERSE = False  #@param {type:"boolean"}
SAVE_STATS = False  #@param {type:"boolean"}
OUT_DIR = "/content/plato_keys_out"  #@param {type:"string"}

eff_batch = min(BATCH_SIZE, 8) if NEED_8BIT else BATCH_SIZE
cmd = [
    "python", "-u", "scripts/calibrate.py",
    "--source", SOURCE_MODEL,
    "--target", TARGET_MODEL,
    "--preset", PRESET,
    "--num-prompts", str(NUM_PROMPTS),
    "--val-prompts", str(VAL_PROMPTS),
    "--batch-size", str(eff_batch),
    "--max-length", str(MAX_LENGTH),
    "--out-dir", OUT_DIR,
]
if NEED_8BIT:
    cmd.append("--target-8bit")
if ALSO_REVERSE:
    cmd.append("--also-reverse")
if SAVE_STATS:
    cmd.append("--save-stats")

print("Running:", " ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
if proc.wait() != 0:
    raise RuntimeError(f"calibrate.py exited with code {proc.returncode}")

In [ ]:
#@title 4. Inspect generated keys
from pathlib import Path

from plato_keys import load_key

for p in sorted(Path(OUT_DIR).glob("*.safetensors")):
    print("=" * 72)
    print(f"File: {p.name} ({p.stat().st_size / 1024 / 1024:.1f} MB)")
    print(load_key(p).summary())

In [ ]:
#@title 5. Download `.safetensors` keys (or copy to Google Drive)
import shutil
import zipfile
from pathlib import Path

SAVE_TO_DRIVE = False  #@param {type:"boolean"}
DRIVE_FOLDER = "/content/drive/MyDrive/PlatoKeys"  #@param {type:"string"}

keys = sorted(Path(OUT_DIR).glob("*.safetensors"))
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = Path(DRIVE_FOLDER)
    dst.mkdir(parents=True, exist_ok=True)
    for k in keys:
        shutil.copy2(k, dst / k.name)
        print("Copied to Drive:", dst / k.name)
else:
    from google.colab import files

    zip_path = "/content/plato_keys_moss_qwen35.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as zf:
        for k in keys:
            zf.write(k, arcname=k.name)
        metrics_file = Path(OUT_DIR) / "metrics.json"
        if metrics_file.exists():
            zf.write(metrics_file, arcname="metrics.json")
    files.download(zip_path)
    print("Drop the .safetensors files into ComfyUI/models/plato_keys/ and restart/refresh ComfyUI.")